# Data Exploration and Feature Engineering

This notebook is for exploring the generated training data and performing feature engineering for ML model training.


In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
print("Libraries imported successfully.")

Libraries imported successfully.


In [16]:
# Boolean flag to control data loading (set True when training data exists in data/training/)
LOAD_DATA = False
PRINT_ON_SCREEN = True

# User IO flags
print('User IO flags set:')
print(f'LOAD_DATA == {LOAD_DATA}')
print(f'PRINT_ON_SCREEN == {PRINT_ON_SCREEN}')

User IO flags set:
LOAD_DATA == False
PRINT_ON_SCREEN == True


In [17]:
# 1. Set up file paths and load data (notebook lives under notebooks/; paths relative to that)
# 2. Load the dataset and metadata (or auto-discover latest when LOAD_DATA=False)
BASE_DIR_ALL = '../data/training'  # from notebooks/ go up one, then data/training
df_data = pd.DataFrame()
df_meta = {}

if LOAD_DATA:
    # User specified: load named files (edit names below if needed)
    CORE_DATA_FILE_NAME = "training_data_complete_20260116_224238.pkl"
    META_DATA_FILE_NAME = "metadata_20260116_224238.json"
    FILE_PATH_DATA = os.path.join(BASE_DIR_ALL, CORE_DATA_FILE_NAME)
    FILE_PATH_META = os.path.join(BASE_DIR_ALL, META_DATA_FILE_NAME)
    if os.path.isfile(FILE_PATH_DATA):
        df_data = pd.read_pickle(FILE_PATH_DATA)
        if os.path.isfile(FILE_PATH_META):
            with open(FILE_PATH_META, 'r') as f:
                df_meta = json.load(f)
        print(f'Loaded {CORE_DATA_FILE_NAME}.')
    else:
        print(f'File not found: {FILE_PATH_DATA}')
else:
    # Auto-discover latest training data (pkl only)
    import glob
    pkls = sorted(glob.glob(os.path.join(BASE_DIR_ALL, "training_data_complete_*.pkl")), reverse=True)
    if pkls:
        df_data = pd.read_pickle(pkls[0])
        stem = os.path.basename(pkls[0]).replace("training_data_complete_", "").replace(".pkl", "")
        meta_path = os.path.join(BASE_DIR_ALL, f"metadata_{stem}.json")
        if os.path.isfile(meta_path):
            with open(meta_path, 'r') as f:
                df_meta = json.load(f)
        print(f'Auto-loaded latest: {os.path.basename(pkls[0])}')
    else:
        print('No training data (.pkl) found in data/training/. Run Main_2_generate_training_data first.')

if len(df_data) > 0:
    print(f"Data shape: {df_data.shape[0]} rows, {df_data.shape[1]} columns")

Auto-loaded latest: training_data_complete_20260116_224238.pkl
Data shape: 152442 rows, 328 columns


In [18]:
# Data exploration (only when we have data)
if len(df_data) == 0:
    print("No data loaded. Skip exploration or load data in the previous cell.")
elif PRINT_ON_SCREEN:
    print("Dataframe shape:", df_data.shape[0], "rows and", df_data.shape[1], "columns")
    print(50*"-")
    if 'reactant_type' in df_data.columns:
        print('Unique reactant types:', df_data['reactant_type'].unique())
    print(50*"-")
    print("Dataframe columns:\n", df_data.columns.to_numpy())
    print(50*"-")
else:
    print("Data exploration output skipped (PRINT_ON_SCREEN=False).")

Dataframe shape: 152442 rows and 328 columns
--------------------------------------------------
Unique reactant types: ['n-Hexane']
--------------------------------------------------
Dataframe columns:
 ['reactant_type' 'initial_temperature_K' 'initial_pressure_Pa'
 'reactor_length_m' 'reactor_diameter_m' 'mass_flow_rate_kgps'
 'heat_flux_Wm2' 'z_position_m' 'relative_position' 'temperature_K'
 'pressure_Pa' 'velocity_ms' 'density_kgm3' 'heat_capacity_cp_JkgK'
 'heat_capacity_cv_JkgK' 'mean_molecular_weight_kgkmol' 'enthalpy_Jkg'
 'entropy_JkgK' 'internal_energy_Jkg' 'gibbs_free_energy_Jkg'
 'viscosity_Pas' 'thermal_conductivity_WmK' 'Y_Water' 'X_Water' 'Y_Ar'
 'X_Ar' 'Y_He' 'X_He' 'Y_Ne' 'X_Ne' 'Y_N2' 'X_N2' 'Y_C6H14(1)'
 'X_C6H14(1)' 'Y_Benzene(2)' 'X_Benzene(2)' 'Y_C8H10(3)' 'X_C8H10(3)'
 'Y_C4H6(4)' 'X_C4H6(4)' 'Y_C4H8(5)' 'X_C4H8(5)' 'Y_Toluene(6)'
 'X_Toluene(6)' 'Y_Styrene(7)' 'X_Styrene(7)' 'Y_C2H4(8)' 'X_C2H4(8)'
 'Y_C3H6(9)' 'X_C3H6(9)' 'Y_C5H6(10)' 'X_C5H6(10)' 'Y_C3H7(13)'


In [19]:
# Organize data into logical categories for better analysis and ML
def organize_data_columns(df):
    """Organize dataframe columns into logical categories."""
    all_cols = df.columns.tolist()
    categories = {
        'inlet_conditions': ['initial_temperature_K', 'initial_pressure_Pa'],
        'reactor_design': ['reactor_length_m', 'reactor_diameter_m'],
        'operating_conditions': ['mass_flow_rate_kgps', 'heat_flux_Wm2'],
        'spatial_coordinates': ['z_position_m', 'relative_position'],
        'state_variables': ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3'],
        'thermodynamic_properties': [
            'heat_capacity_cp_JkgK', 'heat_capacity_cv_JkgK', 'mean_molecular_weight_kgkmol',
            'enthalpy_Jkg', 'entropy_JkgK', 'internal_energy_Jkg', 'gibbs_free_energy_Jkg',
            'viscosity_Pas', 'thermal_conductivity_WmK'
        ],
        'species_mass_fractions': [c for c in all_cols if c.startswith('Y_')],
        'species_mole_fractions': [c for c in all_cols if c.startswith('X_')]
    }
    organized = {}
    for category, cols in categories.items():
        existing = [c for c in cols if c in all_cols]
        if existing:
            organized[category] = existing
    return organized

# Run organization only when we have data
data_categories = {}
df_boundary = df_reactor = df_operating = df_spatial = None
df_state = df_thermo = df_species_Y = df_species_X = df_main = None

if len(df_data) > 0:
    data_categories = organize_data_columns(df_data)
    print("=" * 70)
    print("DATA ORGANIZATION BY CATEGORY")
    print("=" * 70)
    if PRINT_ON_SCREEN:
        for category, cols in data_categories.items():
            print(f"\n{category.replace('_', ' ').title()}: Count: {len(cols)}")
            if len(cols) <= 4:
                print(f"  Columns: {cols}")
            else:
                print(f"  Columns: {cols[:5]} ... ({len(cols) - 5} more)")
    print("=" * 70)
    df_boundary = df_data[data_categories['inlet_conditions']] if 'inlet_conditions' in data_categories else None
    df_reactor = df_data[data_categories['reactor_design']] if 'reactor_design' in data_categories else None
    df_operating = df_data[data_categories['operating_conditions']] if 'operating_conditions' in data_categories else None
    df_spatial = df_data[data_categories['spatial_coordinates']] if 'spatial_coordinates' in data_categories else None
    df_state = df_data[data_categories['state_variables']] if 'state_variables' in data_categories else None
    df_thermo = df_data[data_categories['thermodynamic_properties']] if 'thermodynamic_properties' in data_categories else None
    df_species_Y = df_data[data_categories['species_mass_fractions']] if 'species_mass_fractions' in data_categories else None
    df_species_X = df_data[data_categories['species_mole_fractions']] if 'species_mole_fractions' in data_categories else None
    main_input_features = (
        data_categories.get('inlet_conditions', []) +
        data_categories.get('operating_conditions', []) +
        data_categories.get('reactor_design', []) +
        data_categories.get('spatial_coordinates', []))
    main_input_features = [c for c in main_input_features if c in df_data.columns]
    df_main = df_data[main_input_features] if main_input_features else pd.DataFrame()
else:
    print("No data to organize. Load data in the previous cell.")


DATA ORGANIZATION BY CATEGORY

Inlet Conditions: Count: 2
  Columns: ['initial_temperature_K', 'initial_pressure_Pa']

Reactor Design: Count: 2
  Columns: ['reactor_length_m', 'reactor_diameter_m']

Operating Conditions: Count: 2
  Columns: ['mass_flow_rate_kgps', 'heat_flux_Wm2']

Spatial Coordinates: Count: 2
  Columns: ['z_position_m', 'relative_position']

State Variables: Count: 4
  Columns: ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3']

Thermodynamic Properties: Count: 9
  Columns: ['heat_capacity_cp_JkgK', 'heat_capacity_cv_JkgK', 'mean_molecular_weight_kgkmol', 'enthalpy_Jkg', 'entropy_JkgK'] ... (4 more)

Species Mass Fractions: Count: 153
  Columns: ['Y_Water', 'Y_Ar', 'Y_He', 'Y_Ne', 'Y_N2'] ... (148 more)

Species Mole Fractions: Count: 153
  Columns: ['X_Water', 'X_Ar', 'X_He', 'X_Ne', 'X_N2'] ... (148 more)
